# Task 2: Subindustry Classification (428 classes)
### FLANG-BERT | v9c Sibling Context | Auxiliary Heads | 4CE+4FL+4CE
- **Backbone**: `SALT-NLP/FLANG-BERT`
- **Input**: `[SEG] SegmentName: Description [SIBLINGS] sib1 | sib2`
- **Loss**: multi-class CE (one label per segment, NOT BCE)
- **Aux heads**: industry (145) + sector (11)
- **Expected**: 0.62–0.70 macro-F1 | ~2–3 hrs on A100

## 0. Install

In [ ]:
!pip install -q transformers==4.40.0 accelerate scikit-learn


## 1. Imports

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
from torch.cuda.amp import autocast, GradScaler

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 2. Config

In [ ]:
class Config:
    MODEL_NAME    = 'SALT-NLP/FLANG-BERT'
    MAX_LEN       = 512
    # Updated automatically after LabelEncoder.fit()
    NUM_SUBIND    = 428
    NUM_INDUSTRY  = 145
    NUM_SECTOR    = 11
    # Auxiliary loss weights (proven in Task 1)
    W_SUBIND      = 1.00
    W_INDUSTRY    = 0.15
    W_SECTOR      = 0.20
    # Schedule: 4CE -> 4FL -> 4CE
    CE_EPOCHS_1   = 4
    FL_EPOCHS     = 4
    FL_GAMMA      = 2.0
    CE_EPOCHS_2   = 4
    # Optimiser
    BATCH_SIZE    = 16
    GRAD_ACCUM    = 2
    LR            = 2e-5
    WEIGHT_DECAY  = 0.01
    WARMUP_RATIO  = 0.06
    MAX_GRAD_NORM = 1.0
    # Paths -- update to your Colab paths
    TRAIN_CSV     = '/content/train.csv'
    VAL_CSV       = '/content/val.csv'
    TEST_CSV      = '/content/test.csv'
    OUTPUT_DIR    = '/content/task2_flangbert'
    SEED          = 42

cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)
print('Config OK')


## 3. v9c Text Builder (Sibling Context)

In [ ]:
def build_v9c_text(row, company_df):
    """
    '[SEG] SegmentName: Description [SIBLINGS] sib1 | sib2'
    Sibling names give the model full company scope,
    reducing subindustry confusion for conglomerates.
    """
    text = f"[SEG] {str(row['SegmentName']).strip()}: {str(row['SegmentDescription']).strip()}"
    siblings = company_df[
        (company_df['CompanyId'] == row['CompanyId']) &
        (company_df.index != row.name)
    ]['SegmentName'].dropna().tolist()
    if siblings:
        text += ' [SIBLINGS] ' + ' | '.join(s.strip() for s in siblings[:5])
    return text


def prepare_df(path, is_train=True):
    df = pd.read_csv(path)
    df['text'] = df.apply(lambda r: build_v9c_text(r, df), axis=1)
    if is_train:
        # Derive auxiliary labels from SubIndustry code structure
        df['IndustryCode'] = df['Subindustry'].astype(str).str[:8]
        df['SectorCode']   = df['Subindustry'].astype(str).str[:3]
    return df

train_df = prepare_df(cfg.TRAIN_CSV)
val_df   = prepare_df(cfg.VAL_CSV)
test_df  = prepare_df(cfg.TEST_CSV, is_train=False)

print(f'Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}')
print(f'Unique SubIndustries: {train_df["Subindustry"].nunique()}')
print(train_df['text'].iloc[0][:300])


## 4. Label Encoders + Class Weights

In [ ]:
le_sub = LabelEncoder().fit(train_df['Subindustry'])
le_ind = LabelEncoder().fit(train_df['IndustryCode'])
le_sec = LabelEncoder().fit(train_df['SectorCode'])

cfg.NUM_SUBIND   = len(le_sub.classes_)
cfg.NUM_INDUSTRY = len(le_ind.classes_)
cfg.NUM_SECTOR   = len(le_sec.classes_)
print(f'SubInd: {cfg.NUM_SUBIND} | Ind: {cfg.NUM_INDUSTRY} | Sec: {cfg.NUM_SECTOR}')

for df in [train_df, val_df]:
    df['label_sub'] = le_sub.transform(df['Subindustry'])
    df['label_ind'] = le_ind.transform(df['IndustryCode'])
    df['label_sec'] = le_sec.transform(df['SectorCode'])

# Inverse-frequency weights for subindustry imbalance
counts = np.bincount(train_df['label_sub'], minlength=cfg.NUM_SUBIND).astype(float)
counts = np.where(counts == 0, 1, counts)
w = 1.0 / counts
w = w / w.sum() * cfg.NUM_SUBIND
class_weights_t = torch.tensor(w, dtype=torch.float32).to(device)

vc = train_df['Subindustry'].value_counts()
print(f'Imbalance ratio: {vc.max()/max(vc.min(),1):.0f}x | classes < 10 samples: {(vc<10).sum()}')


## 5. Dataset & Balanced Sampler

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_NAME)

class SegDataset(Dataset):
    def __init__(self, df, tok, max_len, is_test=False):
        self.texts   = df['text'].tolist()
        self.tok     = tok
        self.max_len = max_len
        self.is_test = is_test
        if not is_test:
            self.ls = df['label_sub'].tolist()
            self.li = df['label_ind'].tolist()
            self.lc = df['label_sec'].tolist()

    def __len__(self): return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(self.texts[i], max_length=self.max_len,
                       padding='max_length', truncation=True, return_tensors='pt')
        item = {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0)}
        if not self.is_test:
            item['label_sub'] = torch.tensor(self.ls[i], dtype=torch.long)
            item['label_ind'] = torch.tensor(self.li[i], dtype=torch.long)
            item['label_sec'] = torch.tensor(self.lc[i], dtype=torch.long)
        return item

train_ds = SegDataset(train_df, tokenizer, cfg.MAX_LEN)
val_ds   = SegDataset(val_df,   tokenizer, cfg.MAX_LEN)
test_ds  = SegDataset(test_df,  tokenizer, cfg.MAX_LEN, is_test=True)

# WeightedRandomSampler: oversample rare subindustries per batch
samp_w = w[train_df['label_sub'].values]
sampler = WeightedRandomSampler(torch.tensor(samp_w, dtype=torch.double),
                                len(train_ds), replacement=True)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE,   sampler=sampler,   num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE*2, shuffle=False,     num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE*2, shuffle=False,     num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')


## 6. Model — FLANG-BERT + 3 Heads

In [ ]:
class FlangBERT(nn.Module):
    """
    FLANG-BERT backbone with 3 multi-class CE heads.
    Task 2 = multi-CLASS (one label per segment), not multi-label.
    head_subind   : 428 classes  (primary)
    head_industry : 145 classes  (aux, w=0.15)
    head_sector   :  11 classes  (aux, w=0.20)
    """
    def __init__(self, model_name, n_sub, n_ind, n_sec, dropout=0.1):
        super().__init__()
        self.enc  = AutoModel.from_pretrained(model_name)
        h = self.enc.config.hidden_size
        self.norm = nn.LayerNorm(h)
        self.drop = nn.Dropout(dropout)
        self.h_sub = nn.Linear(h, n_sub)
        self.h_ind = nn.Linear(h, n_ind)
        self.h_sec = nn.Linear(h, n_sec)

    def forward(self, input_ids, attention_mask):
        cls = self.enc(input_ids=input_ids,
                       attention_mask=attention_mask).last_hidden_state[:, 0]
        cls = self.drop(self.norm(cls))
        return self.h_sub(cls), self.h_ind(cls), self.h_sec(cls)

model = FlangBERT(cfg.MODEL_NAME, cfg.NUM_SUBIND, cfg.NUM_INDUSTRY, cfg.NUM_SECTOR).to(device)
print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')


## 7. Loss Functions

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()

ce_fn    = nn.CrossEntropyLoss(weight=class_weights_t)
focal_fn = FocalLoss(gamma=cfg.FL_GAMMA, weight=class_weights_t)
aux_fn   = nn.CrossEntropyLoss()

def total_loss(ls, li, lc, y_s, y_i, y_c, primary):
    return (cfg.W_SUBIND   * primary(ls, y_s) +
            cfg.W_INDUSTRY * aux_fn(li, y_i) +
            cfg.W_SECTOR   * aux_fn(lc, y_c))

print('Loss functions ready.')


## 8. Training Utilities

In [ ]:
def make_opt_sched(model, n_steps, lr):
    no_decay = ['bias', 'LayerNorm.weight']
    params = [
        {'params': [p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         'weight_decay': cfg.WEIGHT_DECAY},
        {'params': [p for n,p in model.named_parameters() if     any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    opt = torch.optim.AdamW(params, lr=lr)
    sched = get_cosine_schedule_with_warmup(
        opt, int(cfg.WARMUP_RATIO*n_steps), n_steps)
    return opt, sched


def train_epoch(model, loader, opt, sched, scaler, loss_fn):
    model.train()
    opt.zero_grad()
    total = 0.0
    for step, b in enumerate(loader):
        ids = b['input_ids'].to(device)
        msk = b['attention_mask'].to(device)
        ys  = b['label_sub'].to(device)
        yi  = b['label_ind'].to(device)
        yc  = b['label_sec'].to(device)
        with autocast():
            ls, li, lc = model(ids, msk)
            loss = total_loss(ls, li, lc, ys, yi, yc, loss_fn) / cfg.GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step+1) % cfg.GRAD_ACCUM == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
            scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
        total += loss.item() * cfg.GRAD_ACCUM
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for b in loader:
        ids = b['input_ids'].to(device)
        msk = b['attention_mask'].to(device)
        with autocast():
            ls, _, _ = model(ids, msk)
        preds.extend(ls.argmax(-1).cpu().numpy())
        labels.extend(b['label_sub'].numpy())
    return f1_score(labels, preds, average='macro', zero_division=0), preds, labels

print('Utilities ready.')


## 9. Phase 1 — CrossEntropy Warm-Up (4 epochs)

In [ ]:
steps1 = (len(train_loader)//cfg.GRAD_ACCUM) * cfg.CE_EPOCHS_1
opt, sched = make_opt_sched(model, steps1, cfg.LR)
scaler = GradScaler()
best_f1, log = 0.0, []

print('=== Phase 1: CE warm-up ===')
for ep in range(cfg.CE_EPOCHS_1):
    loss = train_epoch(model, train_loader, opt, sched, scaler, ce_fn)
    vf1, _, _ = evaluate(model, val_loader)
    log.append({'phase':'CE1','epoch':ep+1,'loss':loss,'val_f1':vf1})
    print(f'  [{ep+1}/{cfg.CE_EPOCHS_1}] loss={loss:.4f}  val_macro_F1={vf1:.4f}')
    if vf1 > best_f1:
        best_f1 = vf1
        torch.save(model.state_dict(), f'{cfg.OUTPUT_DIR}/best.pt')
        print(f'    ✓ New best {best_f1:.4f}')


## 10. Phase 2 — Focal Loss Hard-Mining (4 epochs)

In [ ]:
steps2 = (len(train_loader)//cfg.GRAD_ACCUM) * cfg.FL_EPOCHS
opt, sched = make_opt_sched(model, steps2, cfg.LR * 0.5)
scaler = GradScaler()

print('=== Phase 2: Focal Loss hard-mining ===')
for ep in range(cfg.FL_EPOCHS):
    loss = train_epoch(model, train_loader, opt, sched, scaler, focal_fn)
    vf1, _, _ = evaluate(model, val_loader)
    log.append({'phase':'FL','epoch':ep+1,'loss':loss,'val_f1':vf1})
    print(f'  [{ep+1}/{cfg.FL_EPOCHS}] loss={loss:.4f}  val_macro_F1={vf1:.4f}')
    if vf1 > best_f1:
        best_f1 = vf1
        torch.save(model.state_dict(), f'{cfg.OUTPUT_DIR}/best.pt')
        print(f'    ✓ New best {best_f1:.4f}')


## 11. Phase 3 — CrossEntropy Fine-Tune (4 epochs)

In [ ]:
steps3 = (len(train_loader)//cfg.GRAD_ACCUM) * cfg.CE_EPOCHS_2
opt, sched = make_opt_sched(model, steps3, cfg.LR * 0.3)
scaler = GradScaler()

print('=== Phase 3: CE fine-tune ===')
for ep in range(cfg.CE_EPOCHS_2):
    loss = train_epoch(model, train_loader, opt, sched, scaler, ce_fn)
    vf1, _, _ = evaluate(model, val_loader)
    log.append({'phase':'CE2','epoch':ep+1,'loss':loss,'val_f1':vf1})
    print(f'  [{ep+1}/{cfg.CE_EPOCHS_2}] loss={loss:.4f}  val_macro_F1={vf1:.4f}')
    if vf1 > best_f1:
        best_f1 = vf1
        torch.save(model.state_dict(), f'{cfg.OUTPUT_DIR}/best.pt')
        print(f'    ✓ New best {best_f1:.4f}')


## 12. Final Evaluation + Per-Class Report

In [ ]:
model.load_state_dict(torch.load(f'{cfg.OUTPUT_DIR}/best.pt'))
final_f1, preds, labels = evaluate(model, val_loader)
print(f'Best Val Macro-F1: {final_f1:.4f}')

report = classification_report(labels, preds,
    target_names=le_sub.classes_, zero_division=0, output_dict=True)
rdf = pd.DataFrame(report).T.iloc[:-3]
rdf.to_csv(f'{cfg.OUTPUT_DIR}/per_class_f1.csv')

zero_f1 = (rdf['f1-score'] == 0).sum()
print(f'Zero-F1 classes : {zero_f1} / {cfg.NUM_SUBIND}')
print(f'Max achievable  : {(cfg.NUM_SUBIND - zero_f1)/cfg.NUM_SUBIND:.4f}  (perfect on all others)')

pd.DataFrame(log).to_csv(f'{cfg.OUTPUT_DIR}/training_log.csv', index=False)
print(pd.DataFrame(log).to_string())


## 13. Test Inference → submission.csv

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    out = []
    for b in loader:
        ids = b['input_ids'].to(device)
        msk = b['attention_mask'].to(device)
        with autocast():
            ls, _, _ = model(ids, msk)
        out.extend(ls.argmax(-1).cpu().numpy())
    return out

test_df['PredictedSubindustry'] = le_sub.inverse_transform(predict(model, test_loader))
test_df[['CompanyId','AsOfDate','SegmentName','PredictedSubindustry']].to_csv(
    f'{cfg.OUTPUT_DIR}/submission.csv', index=False)
print(f'Saved submission.csv — {len(test_df):,} rows')
test_df[['CompanyId','SegmentName','PredictedSubindustry']].head(10)


## 14. Diagnostics — Where F1 Is Lost

In [ ]:
print('Bottom 20 classes by F1:')
print(rdf.sort_values('f1-score').head(20)[['f1-score','support']].to_string())

corr = rdf[['f1-score','support']].corr().loc['f1-score','support']
print(f'\nCorr(support, F1) = {corr:.3f}')
if corr > 0.5:
    print('-> Imbalance is the main bottleneck: augment rare classes')
elif corr < 0.3:
    print('-> Label confusion is main bottleneck: review ambiguous subindustry pairs')
else:
    print('-> Both imbalance and label confusion are at play')


## 15. Optional: Temperature Scaling (+0.01-0.02 F1 free)

In [ ]:
@torch.no_grad()
def get_logits(model, loader):
    model.eval()
    lg_list, lb_list = [], []
    for b in loader:
        ids = b['input_ids'].to(device)
        msk = b['attention_mask'].to(device)
        with autocast():
            ls, _, _ = model(ids, msk)
        lg_list.append(ls.cpu())
        lb_list.append(b['label_sub'])
    return torch.cat(lg_list), torch.cat(lb_list)

vl, vlab = get_logits(model, val_loader)
best_T, best_Tf1 = 1.0, 0.0
for T in np.arange(0.5, 3.1, 0.1):
    p = (vl / T).argmax(-1).numpy()
    f = f1_score(vlab.numpy(), p, average='macro', zero_division=0)
    if f > best_Tf1:
        best_T, best_Tf1 = T, f

print(f'Best T={best_T:.2f}  F1 after calibration: {best_Tf1:.4f}  (was {final_f1:.4f})')
print(f'Gain: {best_Tf1-final_f1:+.4f}')
